In [1]:
from utils_phoneme_reco import *
MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"
#MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wavlm_finetuned_big/checkpoint-11000"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cuda"
model = model.to(device)
model.eval()


/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [3]:
from jiwer import process_words
from collections import defaultdict
from pathlib import Path
from praatio import textgrid

audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-PARK"
textgrid_dir = Path("/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-PARK")
ref_files = textgrid_dir.glob("*")
ref_dict = {
    str(f.stem)[:-11]: str(f)
    for f in ref_files
    if str(f.stem).endswith("pr_analyse")
}
patho = "park"

In [4]:
import pickle
alignment_store = {}

for f in ref_dict.keys():
    audio_path = os.path.join(audio_dir, f + ".wav")
    textgrid_path = ref_dict[f]
    
    pred_phonemes, pred_alignments = get_phoneme_alignments_w2v_ctcfa(
            model, processor, audio_path
        )
    ref_alignments = get_reference_alignments_typaloc(textgrid_path)
    
    clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
    clean_ref = clean_alignment_dict(ref_alignments,flag="typaloc", is_hyp=False)
    
    # Fix session offset in ref if needed — no hyp information used
    ref_intervals = get_ref_intervals(clean_ref, audio_path, threshold=0.5)
    hyp_intervals = get_hyp_intervals(clean_hyp)

    
    alignment_store[f] = {
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "ref_seq": extract_phoneme_sequence(clean_ref),
        "hyp_seq": extract_phoneme_sequence(clean_hyp),
    }
    


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/vol/experiments3/imbenamor/TAPAS-FRAIS/src/utils/utils_phoneme_reco.py:593: UserWarning: torchaudio.functional._alignment.forced_align has been deprecated. This deprecation is part of a large re

Maximum timestamp in Textgrid changed from (73.375125) to (73.38)
Maximum timestamp in Textgrid changed from (63.158313) to (63.16)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Maximum timestamp in Textgrid changed from (84.149187) to (84.15)


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


Maximum timestamp in Textgrid changed from (51.827) to (51.83)


In [6]:
with open(f"ctc_results/alignment_w2v_{patho}.pkl", "wb") as f:
    pickle.dump(alignment_store, f)

In [10]:
from metrics_alignment import *
import pickle
patho = "ctrl"
with open(f"ctc_results/alignment_w2v_{patho}.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, f"ctc_results/metrics++_w2v_{patho}.csv", per_phoneme_csv=None)
dict_to_csv(alignment_store, f'pred_w2v_{patho}.csv')
#metrics(alignment_store,f"ctc_results/results_w2v_{patho}.csv")

In [24]:
"""all_pairs = []
ref_inventory=set()
pred_inventory=set()
i=0
total_edits = 0
total_ref_tokens = 0
all_boundary_errors=[]
mid_BE=[]
all_predicted_segments=[]
total_S = total_D = total_I = total_N = 0
total_TP_20 = 0
total_FP_20 = 0
total_FN_20 = 0
for f in ref_dict.keys():
    audio_path = os.path.join(audio_dir, f+".wav")
    textgrid_path = ref_dict[f]
    pred_phonemes, pred_alignments = get_phoneme_alignments(model, processor, audio_path)
    ref_alignments = get_reference_alignments_typaloc(textgrid_path)
    clean_hyp = clean_alignment_dict(pred_alignments,is_hyp=True)
    clean_ref = clean_alignment_dict(ref_alignments,flag="typaloc",is_hyp=False)

    ref_offset = clean_ref[0]["start"]
    hyp_offset = clean_hyp[0]["start"]
    #ref_intervals=clean_ref
    #hyp_intervals=clean_hyp
    
    audio_id = f
    
    
    ref_intervals = [{
        "phoneme": item["phoneme"],
        "start": item["start"] - ref_offset,
        "end": item["end"] - ref_offset} for item in clean_ref]
    #print(ref_intervals)
    hyp_intervals = [{
        "phoneme": item["phoneme"],
        "start": item["start"] - hyp_offset,
        "end": item["end"] - hyp_offset} for item in clean_hyp]
    for seg in hyp_intervals:
        all_predicted_segments.append({
            "audio_id": audio_id,
            "phoneme": seg["phoneme"],
            "start": seg["start"],
            "end": seg["end"],
            "duration": seg["end"] - seg["start"]
        })
    ref_midpoints = [(seg["start"] + seg["end"]) / 2 for seg in ref_intervals]
    pred_midpoints = [  (seg["start"] + seg["end"]) / 2 for seg in hyp_intervals]
    TP, FP, FN = compute_f1_20(ref_midpoints, pred_midpoints)
    total_TP_20 += TP
    total_FP_20 += FP
    total_FN_20 += FN
    # Normalize
    ref_seq = extract_phoneme_sequence(clean_ref)
    hyp_seq = extract_phoneme_sequence(clean_hyp)
    
    pred_inventory.update(hyp_seq)
    
    ref_inventory.update(ref_seq)
    print(ref_inventory - pred_inventory)
    print("--")
    print(pred_inventory - ref_inventory)
    all_pairs.append((ref_seq,hyp_seq))
    
    out = process_words(" ".join(ref_seq), " ".join(hyp_seq))
    total_S += out.substitutions
    total_D += out.deletions
    total_I += out.insertions
    total_N += len(ref_seq)
    
    start_err, end_err, dur_err,mid_err,_ = match_alignments_lev(ref_intervals, hyp_intervals,ref_seq,hyp_seq)

    all_boundary_errors.extend(start_err)
    mid_BE.extend(mid_err)"""
    

Maximum timestamp in Textgrid changed from (121.486875) to (121.49)
{'ɥ'}
--
{'ə'}
Maximum timestamp in Textgrid changed from (94.87675) to (94.88)
{'ɥ'}
--
{'ə'}
{'ɥ'}
--
set()
Maximum timestamp in Textgrid changed from (127.059625) to (127.06)
{'ɥ'}
--
set()
Maximum timestamp in Textgrid changed from (142.477688) to (142.48)
{'ɥ'}
--
set()
{'ɥ'}
--
set()
{'ɥ'}
--
set()
Maximum timestamp in Textgrid changed from (112.198875) to (112.2)
{'ɥ'}
--
set()
